Q2. Consider a simple N-Queens problem. Take a suitable value of N and write a Python code for simulated annealing to solve the problem. As we know, if we take T=0 in SA, it falls back to simple hill climbing search. Modify the SA algorithm in order to make it an HC solution. In the report, write proper theoretical justification in support of your modification (e.g., calculation of the acceptance probability) and the behavioral difference between the two approaches. (Note: No marks will be given for simply implementing the HC solution.)

# N-Queens using Simulated Annealing
This section defines a class NQueensVisual to solve the N-Queens problem using the Simulated Annealing algorithm. It includes methods for calculating cost (number of conflicts), generating neighbors, printing the board, and implementing the Simulated Annealing logic. An example is provided to demonstrate its usage with N=8.

In [1]:
import random
import math

class NQueensVisual:
    def __init__(self, N=8):
        self.N = N
        self.state = [random.randint(0, self.N - 1) for _ in range(self.N)]

    def cost(self, state):
        conflicts = 0
        for i in range(self.N):
            for j in range(i + 1, self.N):
                if state[i] == state[j] or abs(state[i] - state[j]) == abs(i - j):
                    conflicts += 1
        return conflicts

    def get_neighbor(self, state):
        neighbor = list(state)
        col = random.randint(0, self.N - 1)
        old_row = neighbor[col]
        new_row = random.randint(0, self.N - 1)
        while new_row == old_row:
            new_row = random.randint(0, self.N - 1)
        neighbor[col] = new_row
        return neighbor, (col, old_row, new_row)

    def print_board(self, state, iteration, action=None, cost=None):
        print(f"Iteration {iteration}:")
        if action:
            col, old_r, new_r = action
            print(f"  ↳ Moved queen in column {col} from row {old_r} → {new_r}")
        if cost is not None:
            print(f"  ↳ Conflicts: {cost}")
        for r in range(self.N):
            print(' '.join('Q' if state[c] == r else '.' for c in range(self.N)))
        print()

    def simulated_annealing(self, max_steps=1000, initial_temp=50, cooling_rate=0.95):
        current = self.state
        best = list(current)
        best_cost = self.cost(current)
        temperature = initial_temp

        print(f"Max iterations allowed: {max_steps}")
        self.print_board(current, 0, cost=best_cost)

        for step in range(1, max_steps + 1):
            current_cost = self.cost(current)
            if current_cost == 0:
                self.print_board(current, step, cost=current_cost)
                print(f"Solved in {step} iterations!")
                return current

            neighbor, action = self.get_neighbor(current)
            neighbor_cost = self.cost(neighbor)
            delta_e = neighbor_cost - current_cost

            if delta_e < 0 or random.uniform(0, 1) < math.exp(-delta_e / temperature):
                current = neighbor

            if self.cost(current) < best_cost:
                best = list(current)
                best_cost = self.cost(current)

            temperature *= cooling_rate
            self.print_board(current, step, action=action, cost=self.cost(current))

        if best_cost == 0:
            print(f"Solved via best memory in {max_steps} iterations!")
            return best

        print("No solution found within max iterations.")
        return None


In [8]:
N=8
solver = NQueensVisual(N)
solver.simulated_annealing()

Max iterations allowed: 1000
Iteration 0:
  ↳ Conflicts: 11
. . . . . . . .
. Q . . Q . . .
. . . . . . . .
Q . Q Q . . Q .
. . . . . . . .
. . . . . . . Q
. . . . . . . .
. . . . . Q . .

Iteration 1:
  ↳ Moved queen in column 2 from row 3 → 7
  ↳ Conflicts: 9
. . . . . . . .
. Q . . Q . . .
. . . . . . . .
Q . . Q . . Q .
. . . . . . . .
. . . . . . . Q
. . . . . . . .
. . Q . . Q . .

Iteration 2:
  ↳ Moved queen in column 7 from row 5 → 2
  ↳ Conflicts: 10
. . . . . . . .
. Q . . Q . . .
. . . . . . . Q
Q . . Q . . Q .
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . Q . . Q . .

Iteration 3:
  ↳ Moved queen in column 3 from row 3 → 0
  ↳ Conflicts: 10
. . . Q . . . .
. Q . . Q . . .
. . . . . . . Q
Q . . . . . Q .
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . Q . . Q . .

Iteration 4:
  ↳ Moved queen in column 3 from row 0 → 3
  ↳ Conflicts: 10
. . . . . . . .
. Q . . Q . . .
. . . . . . . Q
Q . . Q . . Q .
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . Q . . Q . .



# N-Queens using Hill Climbing
This section defines a class NQueensHillClimbing to solve the N-Queens problem using the Hill Climbing algorithm. It includes methods for calculating cost, printing the board, finding the best neighbor, and implementing the Hill Climbing logic. An example is provided to demonstrate its usage.

In [4]:
import random

class NQueensHillClimbing:
    def __init__(self, N=8):
        self.N = N

    def cost(self, state):
        conflicts = 0
        for i in range(self.N):
            for j in range(i + 1, self.N):
                if state[i] == state[j] or abs(state[i] - state[j]) == abs(i - j):
                    conflicts += 1
        return conflicts

    def print_board(self, state, step, restart, action=None, cost=None):
        print(f"Restart {restart} - Step {step}:")
        if action:
            col, old_r, new_r = action
            print(f"  ↳ Moved queen in column {col} from row {old_r} → {new_r}")
        if cost is not None:
            print(f"  ↳ Conflicts: {cost}")
        for r in range(self.N):
            print(' '.join('Q' if state[c] == r else '.' for c in range(self.N)))
        print()

    def get_best_neighbor(self, state):
        best_neighbor = list(state)
        best_cost = self.cost(state)
        best_move = None

        for col in range(self.N):
            original_row = state[col]
            for row in range(self.N):
                if row == original_row:
                    continue
                neighbor = list(state)
                neighbor[col] = row
                neighbor_cost = self.cost(neighbor)
                if neighbor_cost < best_cost:
                    best_neighbor = neighbor
                    best_cost = neighbor_cost
                    best_move = (col, original_row, row)
        return best_neighbor, best_cost, best_move

    def solve(self, max_steps=100, max_restarts=10):
        best_overall = None
        best_cost = float('inf')

        for restart in range(1, max_restarts + 1):
            current = [random.randint(0, self.N - 1) for _ in range(self.N)]
            current_cost = self.cost(current)
            self.print_board(current, 0, restart, cost=current_cost)

            for step in range(1, max_steps + 1):
                neighbor, neighbor_cost, action = self.get_best_neighbor(current)
                if neighbor_cost >= current_cost:
                    break
                current = neighbor
                current_cost = neighbor_cost
                self.print_board(current, step, restart, action=action, cost=current_cost)

                if current_cost == 0:
                    print(f"Solved in Restart {restart}, Step {step}!")
                    return current

            if current_cost < best_cost:
                best_cost = current_cost
                best_overall = current

        print(f"Best state found with {best_cost} conflicts after {max_restarts} restarts.")
        return best_overall

In [9]:
N=8
solver = NQueensHillClimbing()
solver.solve()

Restart 1 - Step 0:
  ↳ Conflicts: 5
. Q . . . Q . .
. . . . . . . Q
. . . . . . Q .
. . . Q . . . .
. . Q . . . . .
Q . . . . . . .
. . . . Q . . .
. . . . . . . .

Restart 1 - Step 1:
  ↳ Moved queen in column 0 from row 5 → 7
  ↳ Conflicts: 4
. Q . . . Q . .
. . . . . . . Q
. . . . . . Q .
. . . Q . . . .
. . Q . . . . .
. . . . . . . .
. . . . Q . . .
Q . . . . . . .

Restart 1 - Step 2:
  ↳ Moved queen in column 2 from row 4 → 5
  ↳ Conflicts: 3
. Q . . . Q . .
. . . . . . . Q
. . . . . . Q .
. . . Q . . . .
. . . . . . . .
. . Q . . . . .
. . . . Q . . .
Q . . . . . . .

Restart 1 - Step 3:
  ↳ Moved queen in column 0 from row 7 → 4
  ↳ Conflicts: 2
. Q . . . Q . .
. . . . . . . Q
. . . . . . Q .
. . . Q . . . .
Q . . . . . . .
. . Q . . . . .
. . . . Q . . .
. . . . . . . .

Restart 1 - Step 4:
  ↳ Moved queen in column 6 from row 2 → 7
  ↳ Conflicts: 1
. Q . . . Q . .
. . . . . . . Q
. . . . . . . .
. . . Q . . . .
Q . . . . . . .
. . Q . . . . .
. . . . Q . . .
. . . . . . Q .

[2, 5, 3, 1, 7, 4, 6, 0]